In [1]:
import os
import numpy as np
import stagpyviz as spv

# Timeseries

To be able to deal with timeseries i.e., a model evolving in time, we will process several timesteps. There are different approaches that can be taken, but we will use the simplest: define a step range into the [`IOutils`](https://stagpyviz.readthedocs.io/en/latest/api/utils.html#i-o) class.

In our example we will process the steps 30 to 39 of the model ``PJB6_YS1`` and given that we will only save the surface of the model to avoid writing the full volume we will add the argument `prefix="-surface"` which will be used to name the outputs.

*Note*: In this example we will not use the [`Scaling`](https://stagpyviz.readthedocs.io/en/latest/api/scaling.html) functionality.

In [11]:
root   = os.path.join(os.environ["M3D_DIR"], "Stagyy")
mdir   = "PJB6_YS1"
mname  = "PJB6_YS1_Rh32"
# change output directory to your own location, if the directory does not exist, it will be created by IOutils
outdir = os.path.join(os.environ["M3D_DIR"], "Stagyy", "notebook", "output")
pvd_out = mname + ".pvd"

iou = spv.IOutils(
  model_name=mname,
  model_dir=mdir,
  basedir=root,
  pvd=pvd_out,
  prefix="-surface",
  output_dir=outdir,
  output_fields=[
    "temperature",
    "velocity_r",
    "grad_T_r",
    "pressure"
  ],
  step_start=30,
  step_end=39
)

print(iou)

Model: PJB6_YS1_Rh32
Model directory: /mnt/c/Users/jourdon/Documents/Models/Stagyy/PJB6_YS1
Output directory: /mnt/c/Users/jourdon/Documents/Models/Stagyy/notebook/output
Output fields: ['temperature', 'velocity_r', 'grad_T_r', 'pressure']
PVD file: PJB6_YS1_Rh32.pvd
Regions: ['composition']
Reset: False
Prefix: -surface
Step: None
Step start: 30
Step end: 39
Delta step: 1
Surface: True
File list: {'basalt': 'bs', 'composition': 'c', 'density': 'rho', 'divergence': 'div', 'e2': 'ed', 'harzburgite': 'hz', 'nrc': 'nrc', 'pressure': 'vp', 'primordial': 'prm', 'proterozoic': 'prot', 'stress': 'str', 'temperature': ('t', 'T'), 'tracer': 'tra', 'velocity': 'vp', 'viscosity': 'eta', 'vorticity': 'vor', 'topography': 'cs', 'heatflux': 'hf'}



## Generate mesh
Before processing timesteps we create the mesh.
Mesh generation can take some time and given that it is Eulerian there is no necessity to re-generate it at every timestep.

In [12]:
fname = iou.get_field_filename("temperature", 39)
print(fname) 
mesh = spv.YinYangMesh(os.path.join(iou.model_dir, fname))
print(mesh)

Checking for file: /mnt/c/Users/jourdon/Documents/Models/Stagyy/PJB6_YS1/PJB6_YS1_Rh32_t00039
	Found.
/mnt/c/Users/jourdon/Documents/Models/Stagyy/PJB6_YS1/PJB6_YS1_Rh32_t00039
Computed Jacobian matrix of shape (1, 184108, 3, 2) in 0.0434472 seconds
Found 92054 elements with negative orientation
Oriented 92054 elements in 0.076233 seconds
Shell mesh created from points in 1.48408 seconds
Yin-Yang mesh re-constructed in 3.23215 seconds
YinYangMesh (0x7f013c863580)
  N Cells:    11598804
  N Points:   5891584
  X Bounds:   -2.198e+00, 2.198e+00
  Y Bounds:   -2.198e+00, 2.198e+00
  Z Bounds:   -2.198e+00, 2.198e+00
  N Arrays:   0


## Create fields instance
As a helping hand, the function [`fields_instances`](https://stagpyviz.readthedocs.io/en/latest/api/fields.html#stagpyviz.fields_instances) provides a number of pre-defined fields that can be added to the mesh.
Although using it is not mandatory as the [`YinYangMesh`](https://stagpyviz.readthedocs.io/en/latest/api/mesh.html#stagpyviz.YinYangMesh) API contains everything needed to add fields to the mesh, it is highly encouraged to make use of the fields already defined.
Of course if a field is not pre-defined it is always possible to add it to the mesh anyway.

In [13]:
fields_cls = spv.fields_instances(iou, mesh)

## Loop over timesteps

Now we need to loop over timesteps and process the model. 
Here we will use the temperature field as our basename to read the information contained in the StagYY binary output.

In the following example we only process fields we specified in the argument `output_fields` of `IOutils` but of course it is possible to add more fields on the fly, compute some gradients, integral etc. 

### PVD file
To easily read timeseries once the model has been processed we make use of a file format ending with `.pvd`. 
This format is recognized by paraview as a timeseries file. Moreover, this file is structured in such a way that we can also easily retrieve time, step and filemame information if needed.

Note that we named the output timestep files following the pattern:
``` python
outfname = f"step{step:05d}{prefix}.vtu"
```
When dealing with timeseries one should always follow this pattern because the pvd writer is defined with this convention.
By default the pvd writer has no prefix, only the name `stepxxxxx`, the prefix is there to contain anything else the user wants to add to follow its own naming convention and the extension is added at the end.
Therefore when we call the function [`timeseries_write`](https://stagpyviz.readthedocs.io/en/latest/api/utils.html#stagpyviz.timeseries_write) we pass the prefix that will be appended to the step and the extension of the files (here `.vtu`):
```python
spv.timeseries_write(pvd, iou.timeseries, prefix=prefix, extension="vtu")
```
where `prefix` can be break down into `"-{iou.model}{iou.prefix}"` i.e., `"-PJB6_YS1_Rh32-surface"`.

In [16]:
for step in iou.steps_idx:
  # Update the step in the IOutils instance
  iou.step = step
  fname = iou.get_field_filename("temperature", iou.step) 
  print(f"Processing step {step}. Reading binary header from file: {fname}")
  # Add a safeguard to check if the file exists before attempting to read it
  if not os.path.exists(os.path.join(iou.model_dir, fname)):
    print(f"\tFile {fname} not found in {iou.model_dir}. Skipping step {step}.")
    continue

  # Now add fields to the mesh
  for field in iou.output_fields:
    print(f"\tAdding field '{field}' to the mesh.")
    fields_cls[field].add_to_mesh()

  print(f"\tFields added to mesh for step {step}. Writing surface mesh...")
  # keep only the surface for output
  surface_mesh = mesh.surface_mesh
  # copy fields from the original mesh to the surface mesh
  for f in mesh.point_data:
    surface_mesh.point_data[f] = mesh.point_data[f][mesh.surface_idx]
  for f in mesh.cell_data:
    surface_mesh.cell_data[f] = mesh.cell_data[f][mesh.surface_cells]

  # Save to disk
  prefix = f"-{iou.model}{iou.prefix}"
  outfname = f"step{step:05d}{prefix}.vtu"
  surface_mesh.save(os.path.join(iou.output_dir, outfname))

  # Create/append pvd file
  iou.timeseries["step"] = [f"{step:05d}"]
  iou.timeseries["time"] = [iou.time]
  pvd = os.path.join(iou.output_dir, iou.pvd)
  spv.timeseries_write(pvd, iou.timeseries, prefix=prefix, extension="vtu")

  # clear fields from mesh for next step
  for field in iou.output_fields:
    fields_cls[field].reset()


Checking for file: /mnt/c/Users/jourdon/Documents/Models/Stagyy/PJB6_YS1/PJB6_YS1_Rh32_t00030
	Found.
Processing step 30. Reading binary header from file: /mnt/c/Users/jourdon/Documents/Models/Stagyy/PJB6_YS1/PJB6_YS1_Rh32_t00030
	Adding field 'temperature' to the mesh.
Checking for file: /mnt/c/Users/jourdon/Documents/Models/Stagyy/PJB6_YS1/PJB6_YS1_Rh32_t00030
	Found.
Added field temperature to the mesh in 0.117821 seconds
	Adding field 'velocity_r' to the mesh.
Checking for file: /mnt/c/Users/jourdon/Documents/Models/Stagyy/PJB6_YS1/PJB6_YS1_Rh32_vp00030
	Found.
Velocity field reconstructed in 1.10978 seconds
Added field velocity to the mesh in 0.421988 seconds
cartesian to spherical vector transformation performed in 1.81793 seconds
	Adding field 'grad_T_r' to the mesh.
Computed Jacobian matrix of shape (11598804, 3, 3) in 4.32992 seconds
Computed determinant of Jacobian matrix of shape (11598804, 3, 3) in 0.715225 seconds
Computed inverse of Jacobian matrix of shape (11598804, 3, 

## Retrieving information from pvd file

Using the [`timeseries`](https://stagpyviz.readthedocs.io/en/latest/api/utils.html#timeseries) utility we can also retrieve the timeseries information from the pvd file. 
This can be really useful for re-processing files that have already been processed, restarting from an unfinished time loop or whatever we could think about that requires knowledge about timesteps.

Therefore, now that we have written our pvd file, let's inspect it!

In [17]:
pvd_fname = os.path.join(iou.output_dir, iou.pvd)
timeseries = spv.timeseries_process(pvd_fname)
print(timeseries)

{'time': ['0.03960665315389633', '0.0409255288541317', '0.04224745184183121', '0.04356614872813225', '0.04488678276538849', '0.0462074801325798', '0.04752722755074501', '0.0488477423787117', '0.05016887187957764', '0.051487892866134644'], 'step_dir': ['', '', '', '', '', '', '', '', '', ''], 'step': ['30', '31', '32', '33', '34', '35', '36', '37', '38', '39'], 'fname': ['step00030-PJB6_YS1_Rh32-surface.vtu', 'step00031-PJB6_YS1_Rh32-surface.vtu', 'step00032-PJB6_YS1_Rh32-surface.vtu', 'step00033-PJB6_YS1_Rh32-surface.vtu', 'step00034-PJB6_YS1_Rh32-surface.vtu', 'step00035-PJB6_YS1_Rh32-surface.vtu', 'step00036-PJB6_YS1_Rh32-surface.vtu', 'step00037-PJB6_YS1_Rh32-surface.vtu', 'step00038-PJB6_YS1_Rh32-surface.vtu', 'step00039-PJB6_YS1_Rh32-surface.vtu'], 'line': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12]}


We can see that timeseries is a dictionnary containing the time information, a subdirectory if any (this is not the case here), the steps, the files name and the line number in the pvd file of each step.
This last information is mostly used internally and is unlikely to be useful to the user in the general case.

Now that we have these information we can for example loop over the already written files and process them again for some new treatment.
Because they were saved as surface mesh we can load them back as [`ShellMesh`](https://stagpyviz.readthedocs.io/en/latest/api/mesh.html#stagpyviz.ShellMesh). 
The advantage is that now we are working with reduced data so they are much lighter, the disadvantage is that we have lost the information about the volume, so if one needs to access the volume data, they need to start the processing from the beginning (or save the full volume to disk).

As an example let's process the first step we wrote:

In [18]:
fname = timeseries["fname"][0]
surf_mesh = spv.ShellMesh(os.path.join(iou.output_dir, fname))
print(surf_mesh)

Computed Jacobian matrix of shape (1, 184108, 3, 2) in 0.0393925 seconds
Found 0 elements with negative orientation
Oriented 0 elements in 0.0663084 seconds
VTU file loaded in 3.29214 seconds
ShellMesh (0x7f013bf84460)
  N Cells:    184108
  N Points:   92056
  X Bounds:   -2.198e+00, 2.198e+00
  Y Bounds:   -2.198e+00, 2.198e+00
  Z Bounds:   -2.198e+00, 2.198e+00
  N Arrays:   7


In [19]:
print(surf_mesh.point_data)

pyvista DataSetAttributes
Association     : POINT
Active Scalars  : temperature
Active Vectors  : None
Active Texture  : None
Active Normals  : None
Contains arrays :
    temperature             float64    (92056,)             SCALARS
    velocity                float64    (92056, 3)
    velocity_r              float64    (92056, 3)
    pressure                float64    (92056,)


In [9]:
print(surf_mesh.cell_data)

pyvista DataSetAttributes
Association     : CELL
Active Scalars  : neighbors
Active Vectors  : None
Active Texture  : None
Active Normals  : None
Contains arrays :
    neighbors               int32      (184108, 3)          SCALARS
    grad_T                  float64    (184108, 3)
    grad_T_r                float64    (184108, 3)


We have found back the fields we saved on the mesh and we can use them as we want. 